In [1]:
import pandas as pd

from langchain import PromptTemplate
from langchain.chains import LLMChain
from langchain.chat_models import ChatOpenAI

from dotenv import load_dotenv
load_dotenv()

llm = ChatOpenAI(temperature=0, model_name='gpt-3.5-turbo-16k', request_timeout=120) 

/Users/shreyas.sk/PycharmProjects/Skeptic-SEBI/venv/lib/python3.9/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2.0 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
df = pd.read_csv("data/claims_justification_data.csv")

In [3]:
df.head()

,video_id,title,author,description,org_transcript,eng_transcript,image,url,Summary_Claims,Justification
0,flIG8Lw34Cw,D 30 Strategy | Make Daily 5000 to 10k risk Free,Baap of Chart,Today we will learn about the D30 Option Buyi...,वो हार्फ़ेंट कैसे आप लोग अच्छे होंगे आपलोग कमे...,"Text: ""How will you guys be good at Harfent? Y...",<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=flIG8Lw34Cw,The financial influencer claims that with a ca...,The influencer's claim that one can easily mak...
1,ynfEvP5kYCk,Rs 3000 Daily Income from Stock Market | Using...,THE CATALYST GROUP,Earn money daily from stock market | Trick to ...,"Hello everybody, kyaal chaal hai, this is A.S....","Hello everybody, how are you doing? This is A....",<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=ynfEvP5kYCk,"The financial influencer, A.S. Pandit, claims ...",The claim that one can consistently earn a dai...
2,28Jay-3S8fg,Supply Demand Strategy | Trade Swing | Intrada...,Trade Swings,✅ Download NOW ✅ Trade Swings Application (Fre...,दोस्तों आप दिica सब्तू याहापं आप ओंड को बाई कर...,"Friends, you all know that here on Diksha, you...",<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=28Jay-3S8fg,The financial influencer suggests that profits...,The influencer's claim that profits can be mad...
3,BnZU6qYVUl0,99% Accurate | Trade Swing | Intraday Trading ...,Trade Swings,✅ Download NOW ✅ Trade Swings Application (Fre...,"बेन्ग नेफ्टी की यही पढ़टेजिया है, आप देख समकते...",This text seems to be a mixture of Hindi and a...,<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=BnZU6qYVUl0,The financial influencer suggests a trading st...,The influencer's claims are based on his perso...
4,swst4yk-ow8,Bank Bees 100% Risk Free Investment | 1 लाख से...,Dr. Mukul Agrawal,For daily stock market updates join our Telegr...,Hello everyone how are you all and what's goi...,"""Hello everyone, how are you all and what's go...",<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=swst4yk-ow8,"The financial influencer, Mukul Agrawal, promo...",The claim that Bank Bees offers a guaranteed 1...


In [11]:
def assign_gt_labels(sample):
   summary_claims = sample["Summary_Claims"]
   justification = sample["Justification"]

   template="""
      You are a Financial Analyst. Your task is to classify if the Claims and Justification \
      provided on financial information follow entailment, contradiction, or neutrality. \
      Restrict the response to one of the label categories. No need of any explanation

      claim_summary: {summary_claims}
      justifications: {justification}

      Category: 
      """
   prompt = PromptTemplate(
      input_variables=['summary_claims', 'justification'],
      template=template
   )
   chain = LLMChain(llm=llm, prompt=prompt)
   labels = chain.run({'summary_claims':summary_claims, 'justification':justification})
   # formatted_output = parser.parse(labels)

   return labels

In [12]:
df["labels"] = df.apply(assign_gt_labels, axis=1)

In [13]:
df.head(8)

,video_id,title,author,description,org_transcript,eng_transcript,image,url,Summary_Claims,Justification,labels
0,flIG8Lw34Cw,D 30 Strategy | Make Daily 5000 to 10k risk Free,Baap of Chart,Today we will learn about the D30 Option Buyi...,वो हार्फ़ेंट कैसे आप लोग अच्छे होंगे आपलोग कमे...,"Text: ""How will you guys be good at Harfent? Y...",<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=flIG8Lw34Cw,The financial influencer claims that with a ca...,The influencer's claim that one can easily mak...,Contradiction
1,ynfEvP5kYCk,Rs 3000 Daily Income from Stock Market | Using...,THE CATALYST GROUP,Earn money daily from stock market | Trick to ...,"Hello everybody, kyaal chaal hai, this is A.S....","Hello everybody, how are you doing? This is A....",<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=ynfEvP5kYCk,"The financial influencer, A.S. Pandit, claims ...",The claim that one can consistently earn a dai...,Contradiction
2,28Jay-3S8fg,Supply Demand Strategy | Trade Swing | Intrada...,Trade Swings,✅ Download NOW ✅ Trade Swings Application (Fre...,दोस्तों आप दिica सब्तू याहापं आप ओंड को बाई कर...,"Friends, you all know that here on Diksha, you...",<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=28Jay-3S8fg,The financial influencer suggests that profits...,The influencer's claim that profits can be mad...,Contradiction
3,BnZU6qYVUl0,99% Accurate | Trade Swing | Intraday Trading ...,Trade Swings,✅ Download NOW ✅ Trade Swings Application (Fre...,"बेन्ग नेफ्टी की यही पढ़टेजिया है, आप देख समकते...",This text seems to be a mixture of Hindi and a...,<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=BnZU6qYVUl0,The financial influencer suggests a trading st...,The influencer's claims are based on his perso...,neutrality
4,swst4yk-ow8,Bank Bees 100% Risk Free Investment | 1 लाख से...,Dr. Mukul Agrawal,For daily stock market updates join our Telegr...,Hello everyone how are you all and what's goi...,"""Hello everyone, how are you all and what's go...",<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=swst4yk-ow8,"The financial influencer, Mukul Agrawal, promo...",The claim that Bank Bees offers a guaranteed 1...,Contradiction
5,_0fwOm5K7Og,Nifty Best Strategy || 100 % Prob. Of Profit |...,Advance Option Trading,Whatsapp Channel Link \nhttps://wa.me/9170157...,प्राज्टेर भी टेलिगराम के लिए बना था दोस्तों आज...,"""Pragter was also made for Telegram, friends, ...",<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=_0fwOm5K7Og,The financial influencer presents a Nifty intr...,The influencer's claim that there is no risk i...,Contradiction
6,YsHnsJE7CzY,100 % Risk Free Trading Strategy | Earn fix 5...,Stock Market University,#StockMarket #TechnicalAnalysis #Nifty #Financ...,हेलो दोस्तों आप सबी का सुआगत है स्टॉक मार्केट ...,"""Hello friends, welcome to the Stock Market Un...",<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=YsHnsJE7CzY,The financial influencer claims that Nifty Bee...,The claim that Nifty Bees is 110% risk-free is...,Contradiction
7,goSvZXKcLAI,Secret Swing Trading Strategy | 100% Risk free...,Stock Market University,#StockMarket #TechnicalAnalysis #Nifty #Financ...,हेल्लो दोस्तों आप सबी का स्वागत है स्टॉक मार्क...,"""Hello friends, welcome to Stock Market Univer...",<PIL.JpegImagePlugin.JpegImageFile image mode=...,https://www.youtube.com/watch?v=goSvZXKcLAI,The financial influencer claims to have a prof...,The claim of a 90% accuracy rate is misleading...,Contradiction


In [16]:
# df.to_csv("data/claims_justification_labels.csv", index=False)